In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

In [7]:
load_dotenv("./.env")

CONNECTION_STRING = os.environ["BLOB_STORAGE_CONNECTION_STRING"]

In [27]:
def upload_file(local_path: Path, container: str, blob_name: str = None) -> None:
    """Upload a single file to a given container."""
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_name = blob_name or local_path.name

    blob_client = blob_service_client.get_blob_client(container=container, blob=blob_name)

    with open(local_path, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)

    print(f"Uploaded {local_path.name} -> {container}/{blob_name}")


def upload_folder(local_folder: Path, container: str) -> None:
    """Upload all files in a folder to a given container."""
    for file in local_folder.iterdir():
        if file.is_file():
            upload_file(file, container)

def list_all_containers():
    print("All containers:")
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    for container in blob_service_client.list_containers():
        print(f"   {container.name}")

def list_all_files_in_container(container):
    print(f"All files in {container}:")
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(container)
    for blob in container_client.list_blobs():
        print(f"   {blob.name}")

def delete_blob(container: str, blob_name: str) -> None:
    """Delete a single blob from a given container."""
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_client = blob_service_client.get_blob_client(container=container, blob=blob_name)

    try:
        blob_client.delete_blob()
        print(f"Deleted {container}/{blob_name}")
    except Exception as e:
        print(f"Failed to delete {container}/{blob_name}: {e}")



In [32]:
list_all_containers()
list_all_files_in_container("processed")

All containers:
   azure-webjobs-hosts
   azure-webjobs-secrets
   processed
   raw-entsoe
   raw-kaggle
   raw-knmi
   raw-offshore
All files in processed:
   knmi_De_Kooy.csv


In [29]:
upload_file(local_path=Path("datasets_raw/knmi_dataset/De Kooy_235_2021-2030.txt"),container="raw-knmi")

Uploaded De Kooy_235_2021-2030.txt -> raw-knmi/De Kooy_235_2021-2030.txt


In [33]:
delete_blob("processed","knmi_De_Kooy.csv")

Deleted processed/knmi_De_Kooy.csv


In [35]:
upload_folder(Path("datasets_raw/knmi_dataset/"), "raw-knmi")

Uploaded De Kooy_235_2011-2020.txt -> raw-knmi/De Kooy_235_2011-2020.txt
Uploaded De Kooy_235_2021-2030.txt -> raw-knmi/De Kooy_235_2021-2030.txt
Uploaded Hoek van Holland_330_2011-2020.txt -> raw-knmi/Hoek van Holland_330_2011-2020.txt
Uploaded Hoek van Holland_330_2021-2030.txt -> raw-knmi/Hoek van Holland_330_2021-2030.txt
Uploaded Hoorn_251_2011-2020.txt -> raw-knmi/Hoorn_251_2011-2020.txt
Uploaded Hoorn_251_2021-2030.txt -> raw-knmi/Hoorn_251_2021-2030.txt
Uploaded Lauwersoog_277_2011-2020.txt -> raw-knmi/Lauwersoog_277_2011-2020.txt
Uploaded Lauwersoog_277_2021-2030.txt -> raw-knmi/Lauwersoog_277_2021-2030.txt
Uploaded Valkenburg_210_2011-2020.txt -> raw-knmi/Valkenburg_210_2011-2020.txt
Uploaded Vlieland_242_2011-2020.txt -> raw-knmi/Vlieland_242_2011-2020.txt
Uploaded Vlieland_242_2021-2030.txt -> raw-knmi/Vlieland_242_2021-2030.txt
Uploaded Vlissingen_310_2011-2020.txt -> raw-knmi/Vlissingen_310_2011-2020.txt
Uploaded Vlissingen_310_2021-2030.txt -> raw-knmi/Vlissingen_310_202